# CHAMP pristine_v2 只读复核
从项目根目录或本 notebook 所在目录执行。以下单元已作为普通 Python 执行并保存输出；不运行 GPU、训练、删图或修改数据集。

In [1]:
from pathlib import Path
import json
from collections import Counter
ROOT = Path.cwd()
if not (ROOT / 'snapshot.json').exists():
    ROOT = ROOT / 'research/champ_detail_quality_20261025'
snapshot = json.loads((ROOT / 'snapshot.json').read_text(encoding='utf-8-sig'))
summary = json.loads((ROOT / 'inspection/summary.json').read_text(encoding='utf-8'))
records = json.loads((ROOT / 'inspection/records.json').read_text(encoding='utf-8'))
print(snapshot['observed_at'])
print('frames:', len(records), 'sequences:', dict(Counter(row['sequence_id'] for row in records)))

2026-10-25T17:19:12.009835+08:00
frames: 46 sequences: {'000000': 6, '000111': 10, '000444': 19, '000666': 11}


In [2]:
pairs = [json.loads(line) for line in (ROOT / 'source/CHAMP_v1/manifests/pairs/train_pairs_pristine_v2.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
ids = {row['frame_id'] for row in records}
used = {str(p['sequence_id']) + '_' + str(p[side + '_stem']) for p in pairs for side in ['source', 'target']}
assert ids == used
print('pairs:', len(pairs), 'unique frames:', len(used), 'directory match:', ids == used)

pairs: 158 unique frames: 46 directory match: True


In [3]:
import hashlib
for row in snapshot['frames']:
    p = ROOT / 'input/train_rgb_pristine_v2' / (row['frame_id'] + '.png')
    assert hashlib.sha256(p.read_bytes()).hexdigest() == row['sha256'] == row['source_sha256']
print('byte-identical snapshot/source frames:', len(snapshot['frames']))
print('native face width min/median/max:', summary['face_width_min_median_max'])
print('face width 94..106 pixels:', summary['faces_with_width_94_to_106'])
print('hand slots with insufficient legacy keypoints:', summary['unsupported_legacy_hand_metrics'], '/', summary['total_hand_slots'])

byte-identical snapshot/source frames: 46
native face width min/median/max: [94, 143.5, 538]
face width 94..106 pixels: 6
hand slots with insufficient legacy keypoints: 15 / 92


In [4]:
example = next(row for row in records if row['frame_id'] == '000111_00088')
print('example:', example['frame_id'])
print('legacy hand score:', example['legacy_quality']['hands']['left_hand']['lap_var'])
print('native crop:', example['regions']['hand_a']['crop'])
print('A numerical score is not a texture-quality label; view the native crop.')

example: 000111_00088
legacy hand score: 272.83
native crop: native_crops/000111_00088_hand_a.png
A numerical score is not a texture-quality label; view the native crop.
